# Full Dataset EDA

This notebook profiles the full local challenge datasets, including the complete transaction history. It is separate from the 914-row platform submission file.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
DATASETS_DIR = PROJECT_ROOT / 'Datasets'
DOCS_DIR = PROJECT_ROOT / 'Docs'

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

print(PROJECT_ROOT)

## Load Raw Data

In [ ]:
outlets = pd.read_csv(DATASETS_DIR / 'outlet_master.csv')
coords = pd.read_csv(DATASETS_DIR / 'outlet_coordinates.csv')
transactions = pd.read_csv(DATASETS_DIR / 'transactions_history_final.csv')
seasonality = pd.read_csv(DATASETS_DIR / 'distributor_seasonality_details.csv')
holidays = pd.read_csv(DATASETS_DIR / 'holiday_list.csv')

datasets = {
    'outlet_master': outlets,
    'outlet_coordinates': coords,
    'transactions_history': transactions,
    'distributor_seasonality': seasonality,
    'holiday_list': holidays,
}

overview = pd.DataFrame([
    {
        'dataset': name,
        'rows': len(df),
        'columns': len(df.columns),
        'duplicate_rows': int(df.duplicated().sum()),
        'memory_mb': round(df.memory_usage(deep=True).sum() / (1024 * 1024), 2),
    }
    for name, df in datasets.items()
])
overview

## Data Quality Profile

In [ ]:
missing = []
for name, df in datasets.items():
    for column, count in df.isna().sum().items():
        if count:
            missing.append({
                'dataset': name,
                'column': column,
                'missing_rows': int(count),
                'missing_pct': round(100 * count / len(df), 3),
            })
missing = pd.DataFrame(missing)
missing

In [ ]:
outlet_type_map = {'Grocry': 'Grocery', 'Bakry': 'Bakery', 'Eatery ': 'Eatery'}
outlet_size_map = {'small': 'Small', '': 'Unknown'}

outlets_eda = outlets.copy()
outlets_eda['Outlet_Size_Normalized'] = (
    outlets_eda['Outlet_Size'].astype('string').fillna('').str.strip().replace(outlet_size_map).replace('', 'Unknown')
)
outlets_eda['Outlet_Type_Normalized'] = (
    outlets_eda['Outlet_Type'].astype('string').fillna('').str.strip().replace(outlet_type_map)
)

print('Raw outlet sizes')
display(outlets['Outlet_Size'].fillna('<missing>').astype(str).str.strip().value_counts(dropna=False))
print('Raw outlet types')
display(outlets['Outlet_Type'].fillna('<missing>').astype(str).str.strip().value_counts(dropna=False))

In [ ]:
coords_eda = coords.copy()
coords_eda['Latitude'] = pd.to_numeric(coords_eda['Latitude'], errors='coerce')
coords_eda['Longitude'] = pd.to_numeric(coords_eda['Longitude'], errors='coerce')
coords_eda['valid_coordinates'] = coords_eda['Latitude'].between(5.5, 10.2) & coords_eda['Longitude'].between(79.0, 82.1)

coordinate_quality = pd.DataFrame([
    {'metric': 'rows', 'value': len(coords_eda)},
    {'metric': 'unique_outlets', 'value': coords_eda['Outlet_ID'].nunique()},
    {'metric': 'duplicate_outlet_ids', 'value': int(coords_eda.duplicated('Outlet_ID').sum())},
    {'metric': 'valid_coordinate_rows', 'value': int(coords_eda['valid_coordinates'].sum())},
    {'metric': 'invalid_coordinate_rows', 'value': int((~coords_eda['valid_coordinates']).sum())},
    {'metric': 'min_latitude', 'value': coords_eda['Latitude'].min()},
    {'metric': 'max_latitude', 'value': coords_eda['Latitude'].max()},
    {'metric': 'min_longitude', 'value': coords_eda['Longitude'].min()},
    {'metric': 'max_longitude', 'value': coords_eda['Longitude'].max()},
])
coordinate_quality

## Transaction EDA

In [ ]:
tx = transactions.copy()
for column in ['Year', 'Month', 'Volume_Liters', 'Total_Bill_Value']:
    tx[column] = pd.to_numeric(tx[column], errors='coerce')

tx['valid_positive_transaction'] = (tx['Volume_Liters'] > 0) & (tx['Total_Bill_Value'] > 0)
tx_valid = tx.loc[tx['valid_positive_transaction']].copy()
tx_valid['value_per_liter'] = tx_valid['Total_Bill_Value'] / tx_valid['Volume_Liters']

transaction_quality = pd.DataFrame([
    {'metric': 'transaction_rows', 'value': len(tx)},
    {'metric': 'unique_outlets', 'value': tx['Outlet_ID'].nunique()},
    {'metric': 'unique_distributors', 'value': tx['Distributor_ID'].nunique()},
    {'metric': 'unique_skus', 'value': tx['SKU_ID'].nunique()},
    {'metric': 'non_positive_volume_rows', 'value': int((tx['Volume_Liters'] <= 0).sum())},
    {'metric': 'non_positive_bill_rows', 'value': int((tx['Total_Bill_Value'] <= 0).sum())},
    {'metric': 'rows_failing_positive_check', 'value': int((~tx['valid_positive_transaction']).sum())},
    {'metric': 'valid_rows', 'value': len(tx_valid)},
])
transaction_quality

In [ ]:
monthly = tx_valid.groupby(['Outlet_ID', 'Year', 'Month'], as_index=False).agg(
    Monthly_Liters=('Volume_Liters', 'sum'),
    Monthly_Bill_Value=('Total_Bill_Value', 'sum'),
    SKU_Count=('SKU_ID', 'nunique'),
    Transaction_Lines=('SKU_ID', 'size'),
    Distributor_ID=('Distributor_ID', lambda values: values.mode().iat[0]),
)
monthly['Value_Per_Liter'] = monthly['Monthly_Bill_Value'] / monthly['Monthly_Liters']

print('Transaction volume distribution')
display(tx_valid['Volume_Liters'].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]))
print('Outlet-month volume distribution')
display(monthly['Monthly_Liters'].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]))

In [ ]:
monthly_totals = tx_valid.groupby(['Year', 'Month'], as_index=False).agg(
    transaction_rows=('Outlet_ID', 'size'),
    active_outlets=('Outlet_ID', 'nunique'),
    total_liters=('Volume_Liters', 'sum'),
    total_bill_value=('Total_Bill_Value', 'sum'),
)
monthly_totals['avg_liters_per_active_outlet'] = monthly_totals['total_liters'] / monthly_totals['active_outlets']
monthly_totals.round(3)

In [ ]:
distributor_perf = tx_valid.groupby('Distributor_ID', as_index=False).agg(
    transaction_rows=('Outlet_ID', 'size'),
    active_outlets=('Outlet_ID', 'nunique'),
    total_liters=('Volume_Liters', 'sum'),
    avg_transaction_liters=('Volume_Liters', 'mean'),
).sort_values('total_liters', ascending=False)

sku_perf = tx_valid.groupby('SKU_ID', as_index=False).agg(
    transaction_rows=('Outlet_ID', 'size'),
    active_outlets=('Outlet_ID', 'nunique'),
    total_liters=('Volume_Liters', 'sum'),
    avg_value_per_liter=('value_per_liter', 'mean'),
).sort_values('total_liters', ascending=False)

display(distributor_perf.round(3))
display(sku_perf.round(3))

## Outlet-Level Business Cuts

In [ ]:
outlet_sales = monthly.groupby('Outlet_ID').agg(
    active_months=('Monthly_Liters', 'size'),
    mean_monthly_liters=('Monthly_Liters', 'mean'),
    median_monthly_liters=('Monthly_Liters', 'median'),
    max_monthly_liters=('Monthly_Liters', 'max'),
    total_liters=('Monthly_Liters', 'sum'),
    mean_sku_count=('SKU_Count', 'mean'),
    mean_value_per_liter=('Value_Per_Liter', 'mean'),
).reset_index()

outlet_profile = (
    outlets_eda
    .merge(coords_eda[['Outlet_ID', 'valid_coordinates']], on='Outlet_ID', how='left')
    .merge(outlet_sales, on='Outlet_ID', how='left')
)

by_size = outlet_profile.groupby('Outlet_Size_Normalized').agg(
    outlets=('Outlet_ID', 'count'),
    avg_coolers=('Cooler_Count', 'mean'),
    zero_cooler_pct=('Cooler_Count', lambda s: 100 * (s == 0).mean()),
    avg_max_monthly_liters=('max_monthly_liters', 'mean'),
    median_max_monthly_liters=('max_monthly_liters', 'median'),
).round(3)

by_type = outlet_profile.groupby('Outlet_Type_Normalized').agg(
    outlets=('Outlet_ID', 'count'),
    avg_coolers=('Cooler_Count', 'mean'),
    avg_max_monthly_liters=('max_monthly_liters', 'mean'),
    median_max_monthly_liters=('max_monthly_liters', 'median'),
).round(3).sort_values('avg_max_monthly_liters', ascending=False)

display(by_size)
display(by_type)

## Calendar and Seasonality

In [ ]:
holiday_profile = holidays.copy()
holiday_profile['Date'] = pd.to_datetime(holiday_profile['Date'], errors='coerce')
holiday_summary = pd.DataFrame([
    {'metric': 'holiday_rows', 'value': len(holiday_profile)},
    {'metric': 'exact_duplicate_rows', 'value': int(holiday_profile.duplicated().sum())},
    {'metric': 'duplicate_date_name_type_rows', 'value': int(holiday_profile.duplicated(['Date', 'Holiday_Name', 'Holiday_Type']).sum())},
    {'metric': 'unique_dates', 'value': holiday_profile['Date'].nunique()},
])

display(seasonality['Seasonality_Index'].value_counts())
display(holiday_summary)
display(holiday_profile['Holiday_Type'].value_counts())

## Main EDA Conclusions

- The raw data has 20,000 outlets and 2,376,389 transaction rows across 36 months.
- Outlet master contains clear legacy artifacts: missing sizes, lowercase `small`, `Grocry`, and `Bakry`.
- 240 coordinate rows fall outside plausible Sri Lankan bounds and should be quarantined before geospatial work.
- 4,853 transaction rows fail the positive volume or positive bill-value check.
- The wide gap between median and upper-percentile outlet monthly maxima supports a peer-frontier approach for latent potential.
- Holiday rows contain duplicates, so calendar features should be aggregated carefully.

A written summary of this EDA is available in `Docs/eda_summary.md`.